## TEST ALL FILES IN THIS NOTEBOOK

In [ ]:
# [304] clear-f444w  → red
# [305] clear-f150w  → blue
# [306] clear-f200w  → green

from pathlib import Path
from astropy.io import fits
from astroquery.mast import Observations
from astropy.coordinates import SkyCoord
import astropy.units as u

obs_sn = Observations.query_criteria(
    coordinates    = SkyCoord("05h35m28s -69d16m11s", frame="icrs"),
    radius         = 0.05 * u.deg,
    obs_collection = "JWST",
    instrument_name= "NIRCAM/IMAGE",
    calib_level    = 3,
)

products_sn = Observations.get_product_list(obs_sn)
i2d_sn = Observations.filter_products(
    products_sn,
    productSubGroupDescription="I2D",
    extension="fits",
)

manifest = Observations.download_products(
    i2d_sn[[304, 305, 306]],
    download_dir="./jwst_data/sn1987a"
)

# Robust path finder
paths = []
for raw in manifest["Local Path"]:
    p = Path(str(raw))
    if p.exists():
        paths.append(p)
    else:
        matches = list(Path("./jwst_data/sn1987a").rglob(Path(raw).name))
        if matches:
            paths.append(matches[0])
            
print(f"\nDownloaded {len(paths)} files:")
for p in paths:
    with fits.open(p) as hdul:
        phdr = hdul[0].header
        print(f"{p.name}")
        print(f"FILTER  : {phdr.get('FILTER')}")
        print(f"TARGNAME: {phdr.get('TARGNAME')}")
        print(f"EXPTIME : {phdr.get('XPOSURE', phdr.get('EFFEXPTM')):.1f}s")


Downloaded 0 files:


In [ ]:
import numpy as np
from pathlib import Path
from astropy.io import fits
from astropy.wcs import WCS
import jwst_acquire

def load_fits_fixed(path):
    with fits.open(path) as hdul:
        sci = hdul["SCI"].data.astype(np.float32)
        header = hdul["SCI"].header
        wcs = WCS(header)
        phdr = hdul[0].header

        err = hdul["ERR"].data.astype(np.float32) if "ERR" in hdul else np.zeros_like(sci)
        dq = hdul["DQ"].data                     if "DQ"  in hdul else np.zeros_like(sci, dtype=np.uint32)

        filt = phdr.get("FILTER", header.get("FILTER", "UNKNOWN"))
        pupil = phdr.get("PUPIL", "CLEAR")
        targ_name = phdr.get("TARGNAME", "unknown")
        exptime   = phdr.get("XPOSURE", phdr.get("EFFEXPTM", float("nan")))

    # If real filter is on the pupil wheel (e.g. narrowband), use that
    filter_name = pupil if pupil not in ("CLEAR", "NRC_CLEAR", "UNKNOWN", None) else filt

    bad = (dq != 0) | ~np.isfinite(sci)
    sci[bad] = np.nan
    err[bad] = np.nan

    print(f"{Path(path).name}")
    print(f"Filter  : {filter_name}")
    print(f"Shape   : {sci.shape}")
    print(f"Exptime : {exptime}")
    print(f"Range   : {np.nanmin(sci):.4f} – {np.nanmax(sci):.4f} MJy/sr")
    print(f"Bad px  : {bad.sum():,} / {sci.size:,} ({100*bad.sum()/sci.size:.1f}%)")

    return {
        "sci":sci,
        "err":err,
        "wcs":wcs,
        "header":header,
        "path":Path(path),
        "filter":filter_name.upper(),
        "target":targ_name,
        "exptime": exptime,
    }

# Patch into the module
jwst_acquire.load_fits = load_fits_fixed


Patched OK


In [3]:
from align_and_merge import align_and_merge, plot_alignment_result
result = align_and_merge(
    bands,
    blue_filter  = "F115W",
    green_filter = "F277W",
    red_filter   = "F444W",
    use_orb      = True,
)

plot_alignment_result(result)


Aligning F115W → F444W...


c:\Users\shrey\Desktop\CV_JWST_imagery\align_and_merge.py:30: RuntimeWarning: invalid value encountered in cast
  return (scaled * 255).astype(np.uint8)


  Keypoints — reference: 5000  moving: 5000
  Matches: 1555 total → 233 kept
  Homography inliers: 157 / 233
  ORB OK

Aligning F277W → F444W...
  Keypoints — reference: 5000  moving: 5000
  Matches: 2039 total → 305 kept
  Homography inliers: 282 / 305
  ORB OK

Wavelet merging...
  Merged shape: (2108, 2119)
Saved -> alignment_result.png
